## Transfer Learning Project: Fine-Tune a Pretrained Model

In L12, you saw that a pretrained ResNet-18 crushed the bird dataset with minimal training - and you understood why. Pretrained features (edges, textures, shapes learned from ImageNet) transfer to new tasks, especially when your dataset is small.

Now it's your turn. Pick a dataset, fine-tune a pretrained model, and compare strategies. The goal isn't just accuracy - it's understanding *when and why* each strategy works. You'll compare from-scratch training against feature extraction and full fine-tuning on the same data.

In [ ]:
# Setup - run this first
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

## Pick a Dataset

All datasets below are available via `torchvision.datasets` - one line to download, no signup needed.

### EuroSAT (recommended)

**What it is:** 27,000 satellite images (64x64, RGB) of 10 land use classes: annual crop, forest, herbaceous vegetation, highway, industrial buildings, pasture, permanent crop, residential buildings, river, sea/lake.

**Loading:** `torchvision.datasets.EuroSAT(root='./data', download=True)`

| Pros | Cons |
|------|------|
| Interesting domain - satellite imagery is very different from ImageNet's natural photos | Images are small (64x64) - need to resize up for pretrained models |
| 10 classes, ~2,700 per class - good balance | Some classes look similar from above (crop types) |
| Transfer learning payoff is clear: can ImageNet features help with satellite data? | Less visually dramatic than animals/food |
| Fast training due to small source images | |

### FGVCAircraft

**What it is:** 10,000 images of aircraft, labeled by manufacturer (30 classes) or variant (100 classes). Fine-grained classification.

**Loading:** `torchvision.datasets.FGVCAircraft(root='./data', split='trainval', annotation_level='manufacturer', download=True)`

| Pros | Cons |
|------|------|
| Fine-grained classification is genuinely hard from scratch | 30 classes is a lot |
| Very practical - subtle visual differences between airframes | Some manufacturers have few examples |
| Transfer learning is essential here | Requires some domain interest |

### DTD (Describable Textures Dataset)

**What it is:** 5,640 images of 47 texture categories: banded, braided, cracked, dotted, knitted, striped, woven, etc.

**Loading:** `torchvision.datasets.DTD(root='./data', download=True)`

| Pros | Cons |
|------|------|
| Unusual and interesting domain | 47 classes is above ideal - consider subsetting to 10-15 |
| Textures are exactly what pretrained CNNs capture well | ~120 images per class is small |
| Makes a strong case for transfer learning | Some categories are ambiguous |

### Flowers102

**What it is:** 8,189 images of 102 flower species. Used in L12's pretraining-source experiment.

**Loading:** `torchvision.datasets.Flowers102(root='./data', download=True)`

| Pros | Cons |
|------|------|
| 102 fine-grained classes - challenging | Very many classes |
| Beautiful images | You've already seen results in L12 |
| Transfer learning payoff is massive | Training split is small (~10 per class) |

## Recommendations

### Top pick: EuroSAT

The satellite domain is different enough from ImageNet that the transfer learning question is genuinely interesting: can features learned from cats and dogs help classify satellite images? (Yes, surprisingly well.) 10 classes is a sweet spot, training is fast, and the 64x64 images force you to think about input size requirements for pretrained models.

**Suggested approach:** Use all 27,000 images, or subsample to ~5,000 (500 per class) if you want faster iteration.

### Alternative: FGVCAircraft at manufacturer level

If you want a harder fine-grained challenge. 30 classes with subtle visual differences between aircraft manufacturers.

## Project Tasks

Work through these phases in order. Each builds on the previous.

### Phase 1: Data Exploration

*Goal: Understand what you're working with.*

**Task 1.1:** Download your dataset and inspect its structure. How many images? How many classes? What are the class names?

**Task 1.2:** Visualize a grid of sample images. Are the images clean? What resolution? Color or grayscale?

**Task 1.3:** Check class balance. Are all classes equally represented? Plot the distribution.

### Phase 2: Data Pipeline

*Goal: Build DataLoaders that work with pretrained models.*

**Task 2.1:** Set up transforms. Remember: pretrained models expect specific input.
- Resize to 224x224 (even if your images are smaller - pretrained models were trained at this size)
- Use ImageNet normalization: `mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]`
- Add augmentation for training (flips, rotations, color jitter) - keep validation clean

**Task 2.2:** Create train/val split if your dataset doesn't have one. Use `torch.utils.data.random_split` or manual splitting.

**Task 2.3:** Create DataLoaders. Verify a batch has the expected shape.

### Phase 3: From-Scratch Baseline

*Goal: Establish how hard the task is without transfer learning.*

**Task 3.1:** Load ResNet-18 with `weights=None` (random initialization). Replace the final layer for your number of classes.

**Task 3.2:** Train for 5-10 epochs. Record the best validation accuracy. This is your baseline - the accuracy you'd get without any pretrained knowledge.

### Phase 4: Feature Extraction (Frozen Backbone)

*Goal: See what pretrained features alone can do.*

**Task 4.1:** Load ResNet-18 with pretrained ImageNet weights. Freeze all parameters (`requires_grad = False`). Replace the final layer.

**Task 4.2:** Train for 5-10 epochs (only the head is learning). How does this compare to from-scratch? How many trainable parameters now vs. before?

### Phase 5: Full Fine-Tuning

*Goal: Push accuracy higher by adapting the pretrained features.*

**Task 5.1:** Starting from your best frozen-backbone model, unfreeze all layers. Use a lower learning rate than Phase 4 (e.g. 10x smaller).

**Task 5.2:** Train for 5-10 more epochs. Does accuracy improve beyond the frozen model?

**Task 5.3 (optional):** Try discriminative learning rates - lower LR for early layers, higher for later layers and the head. Does it help?

### Phase 6: Evaluation and Comparison

*Goal: Understand what each strategy achieved and why.*

**Task 6.1:** Plot learning curves (train/val accuracy) for all three approaches on the same chart.

**Task 6.2:** Build a confusion matrix for your best model. Which classes are most confused?

**Task 6.3:** Build a comparison table:

| Strategy | Trainable Params | Best Val Accuracy | Training Time |
| --- | --- | --- | --- |
| From scratch | | | |
| Frozen backbone | | | |
| Full fine-tuning | | | |

**Task 6.4:** Reflect - which strategy worked best for your dataset? Why do you think that is? Would you expect the same ranking on a larger dataset? On a dataset very different from ImageNet?

<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
&copy; 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>